<a href="https://colab.research.google.com/github/kasettisiva/R_training_summer_workshop_2026/blob/main/notebooks/03_Data_Structures_and_Wrangling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">

#
# **LONI Scientific Computing Bootcamp 2026**

## **Introduction to R**

*A hands-on workshop for data analysis, visualization, and statistical computing.*

---

**Siva Prasad Kasetti**

**LSU/LONI HPC User Services**  
**skasetti1@lsu.edu**  
**08 June 2026**

</div>

# **Notebook 3: Data Structures & Data Wrangling**

This notebook covers the core R data structures (vectors, data frames, subsetting) and hands-on data wrangling with `dplyr`.

**Sections:** Vectors · Data Frames · Subsetting · Data Cleaning · Data Manipulation · dplyr verbs · Pipe operator

---

# **3 · Data Structures That You'll Actually Use**

Now let's look at the data objects in R. They are:

* Vectors: elements of same class, one dimension
* Matrices: elements of same class, two dimensions
* Arrays: elements of same class, 2+ dimensions
* Lists: elements can be any objects
* Data frames: “datasets” where columns are variables and rows are observations

> We focus on the two you'll encounter 95% of the time:
**vectors** and **data frames**.

### **3.1 Vectors**
Vectors are one-dimentional arrays that contain
elements of the **same** data type.

Vectors can be constructed by
* The `c()` funtion (concatenate):

In [ ]:
# Numeric vector — e.g. expression values for 5 genes
expr <- c(12.4, 0.3, 88.1, 45.0, 7.7)
expr

# Character vector — gene names
genes <- c("BRCA1", "TP53", "MYC", "EGFR", "KRAS")
genes

# Logical vector — which genes are highly expressed (> 40)?
high_expr <- expr > 40
high_expr

# Use that logical vector to filter
genes[high_expr]   # <- This pattern is everywhere in R

### **3.2 Data Frames — the workhorse of R**

Data frames are used to store tabular data.
* They are a special type of **lists**, where each element is a R vector ("column" or "variable") and has to be of the same length.
* The elements (columns) can be of different classes.
* Data frames have special attributes such as row.names.
* Data frames can be created by reading data files, using functions such as `read.table()` or `read.csv()`.


Data frames can be created directly by using the `data.frame()` function.

In [ ]:
# Build a small data frame from scratch
gene_df <- data.frame(
  gene    = genes,
  expr    = expr,
  is_high = high_expr,
  stringsAsFactors = FALSE
)

gene_df

# Basic inspection
nrow(gene_df)
ncol(gene_df)
str(gene_df)

---

# **4· Data Wrangling with Real Data**

Data wrangling means preparing and transforming data so it is easier to analyze.

In real data analysis, we often need to:

- select specific rows or columns
- filter data based on conditions
- check missing values and duplicates
- create new columns
- sort data
- summarize data

We'll use the built-in `mtcars` dataset (car performance specs) — small enough to understand
instantly, realistic enough to show real patterns.


In [ ]:
# Load and preview
data(mtcars)
head(mtcars)

### **Data Cleaning**

Data cleaning focuses on fixing problems in the data.

Examples:

- checking missing values
- removing duplicate rows
- renaming columns
- fixing incorrect values

**Missing Values**

Missing values (missing data) are a common problem in data science. It can have a significant effect on the conclusions that can be drawn from the data. Therefore, R has many functions and packages that deal with the missing value problem.

For example, the `complete.cases` function will scan a dataframe and return a logical vector where the rows without missing values are TRUE and those with missing values FALSE.

In [ ]:
# In the airquality dataset we have used for Exercise 1, there are some missing values.
complete.cases(airquality)

The simplest way of dealing with missing values is to drop all rows with missing values (not necessarily the best way!).

In [ ]:
airquality[complete.cases(airquality),]

**Duplicate Values**

In [ ]:
duplicated(airquality)
airquality[!duplicated(airquality), ]

### **Data Manipulation**

Data manipulation focuses on selecting, changing, organizing, and summarizing data.

Examples:

- creating new columns
- subsetting
- filtering rows
- selecting columns
- sorting rows
- summarizing data

**Creating New Columns**

To add columns/variables to a dataframe, we can simply use the assignment operation.

In [ ]:
gene_df$expr_category <- ifelse(gene_df$expr > 40, "High", "Low")
gene_df

In [ ]:
gene_df$very_high <- gene_df$expr > 80
gene_df

In [ ]:
gene_df$expr_norm <- gene_df$expr / max(gene_df$expr)
gene_df

**Subsetting**

Subsetting means selecting specific parts of your data like a vector, data frame, or list in R.

You can subset:

* By column name
* By position `[row, col]`
* By condition

In [ ]:
# 1. By column name (most readable)
gene_df$expr

# 2. By position [row, col]
gene_df[1, ]      # first row
gene_df[, 2]      # second column

# 3. By condition
gene_df[gene_df$expr > 40, ]

We'll use **dplyr** — the tidyverse data wrangling package.
Its verbs (`filter`, `select`, `mutate`, `summarize`, `arrange`) map almost directly
to SQL.

In [ ]:
# --- dplyr verbs ---

# filter(): keep rows matching a condition
efficient <- filter(mtcars, mpg > 25)
efficient

# select(): keep only certain columns
select(mtcars, mpg, cyl, hp)

# mutate(): add or transform a column
mtcars <- mutate(mtcars, kpl = mpg * 0.425)  # miles/gallon → km/litre
head(mtcars[, c("mpg", "kpl")])

In [ ]:
# arrange(): sort rows
arrange(mtcars, desc(mpg))   # best fuel efficiency first

**The pipe `%>%` (or `|>`)**

The pipe chains operations left-to-right — reads like a recipe.

```r
data %>%
  step1() %>%
  step2() %>%
  step3()
```

> Read them top-to-bottom: each step's output becomes the next step's input.

In [ ]:
# summarize() + group_by(): aggregate statistics by group
mtcars %>%
  group_by(cyl) %>%
  summarize(
    n         = n(),
    mean_mpg  = round(mean(mpg), 1),
    mean_hp   = round(mean(hp), 1),
    mean_wt   = round(mean(wt), 2)
  )

**Quick exercise**

> Using the code above as a template, find the average horsepower (`hp`) and weight (`wt`)
for cars with 4 and 6 cylinders only (exclude `cyl == 8`).

In [ ]:
# Your code here
# Hint: chain filter() before group_by()

In [ ]:
#@title Solution { display-mode: "form" }
mtcars %>%
  filter(cyl != 8) %>%
  group_by(cyl) %>%
  summarize(
    n        = n(),
    mean_mpg = round(mean(mpg), 1),
    mean_hp  = round(mean(hp), 1),
    mean_wt  = round(mean(wt), 2)
  )